In [1]:
from __future__ import annotations
import sys, math, os, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [5]:
# ── Paths ────────────────────────────────────────────────────────────────────
REPO_ROOT = Path("/home/jovyan/cost-based-pruning-layer-by-layer/pruning")
BASELINE_CKPT = Path("/home/jovyan/cost-based-pruning-layer-by-layer/weights/unet_carvana_scale0.5_epoch2.pth")

# config/ sits one level up from pruning/
CONFIG_DIR = REPO_ROOT.parent / "configs"
BASELINE_CHANNELS_TXT = CONFIG_DIR / "model_channels_baseline.txt"

if not BASELINE_CHANNELS_TXT.exists():
    raise FileNotFoundError(f"Cannot find baseline channels file at: {BASELINE_CHANNELS_TXT}")

# ── Add repo to path and import UNet ─────────────────────────────────────────
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.unet_flexible import UNet   # or wherever your UNet class lives

# ── FLOPs / analysis flags ───────────────────────────────────────────────────
USE_2X_MACS  = True    # FLOPs = 2 × MACs (standard convention)
EXCLUDE_OUTC = True    # exclude outc.conv from pruning and FLOPs (matches your codebase)

# ── Input resolution ─────────────────────────────────────────────────────────
H = W = 512            # match whatever you trained/profiled at

In [15]:
# Cell — Theoretical pruning analysis: min keep / max prune per layer, params & FLOPs (min/max)

from __future__ import annotations
import sys, math, os, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ── Config ───────────────────────────────────────────────────────────────────
MIN_KEEP_RATIO  = 0.0
MIN_KEEP_ABS    = 1
FRIENDLY_MULT   = 16
FRIENDLY_ROUND  = "up"   # "up" → ceil to multiple; "down" → floor



USE_2X_MACS_  = True
EXCLUDE_OUTC_ = True
_H = _W = 512

# ── Helpers ───────────────────────────────────────────────────────────────────
def ceil_to_mult(x: int, m: int) -> int:
    return int(math.ceil(x / m) * m)

def floor_to_mult(x: int, m: int) -> int:
    return int(math.floor(x / m) * m)

def _safe_load(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")

def _extract_sd(obj):
    if isinstance(obj, dict):
        for k in ["state_dict", "model_state_dict", "net", "model"]:
            if k in obj and isinstance(obj[k], dict):
                return obj[k]
        return obj
    raise TypeError(type(obj))

def _strip_module(sd: dict) -> dict:
    if any(k.startswith("module.") for k in sd):
        return {k.replace("module.", "", 1): v for k, v in sd.items()}
    return sd

def _infer_n_classes(sd: dict) -> int:
    if "outc.conv.weight" in sd:
        return int(sd["outc.conv.weight"].shape[0])
    for k, v in sd.items():
        if isinstance(v, torch.Tensor) and v.ndim == 4 and v.shape[2] == 1 and "out" in k.lower():
            return int(v.shape[0])
    raise RuntimeError("Cannot infer n_classes")

# ── Load baseline model ───────────────────────────────────────────────────────
raw  = _safe_load(Path(BASELINE_CKPT))
sd   = _strip_module(_extract_sd(raw))
n_cl = _infer_n_classes(sd)

m_base = UNet(n_channels=3, n_classes=n_cl, f_channels=str(BASELINE_CHANNELS_TXT))
m_base.load_state_dict(sd, strict=True)
m_base.eval()

# ── Group / layer-type helpers ────────────────────────────────────────────────
def _group_of(name: str) -> str:
    if name.startswith("outc."):                            return "other"
    if name.startswith("inc."):                             return "encoder"
    if name.startswith(("down1.", "down2.", "down3.")):     return "encoder"
    if name.startswith("down4."):                           return "bottleneck"
    if name.startswith(("up1.", "up2.", "up3.", "up4.")):   return "decoder"
    return "other"

def _out_channels(mod) -> int:
    if isinstance(mod, nn.Conv2d):
        return int(mod.out_channels)
    g = int(getattr(mod, "groups", 1))
    return int(mod.weight.shape[1] * g)

def _is_unpruned_transpose(name: str, mod: nn.Module) -> bool:
    # up1.up / up2.up / up3.up / up4.up are ConvTranspose2d used only for
    # upsampling — they are reshaped to match adjacent layers but never
    # independently pruned in the codebase.
    return isinstance(mod, nn.ConvTranspose2d) and name.endswith(".up")

# ── Gather conv layers ────────────────────────────────────────────────────────
conv_layers = [
    (nm, mod)
    for nm, mod in m_base.named_modules()
    if isinstance(mod, (nn.Conv2d, nn.ConvTranspose2d))
]

# ── Per-layer min-keep / max-prune ────────────────────────────────────────────
records = []
for nm, mod in conv_layers:
    out_ch          = _out_channels(mod)
    grp             = _group_of(nm)
    is_transpose_up = _is_unpruned_transpose(nm, mod)
    is_outc         = EXCLUDE_OUTC_ and nm.endswith("outc.conv")
    is_prunable     = not (is_transpose_up or is_outc)

    if not is_prunable:
        min_keep  = out_ch
        max_prune = 0
        note      = "unpruned-transpose" if is_transpose_up else "outc"
    else:
        by_ratio = int(math.ceil(MIN_KEEP_RATIO * out_ch))
        raw_keep = max(by_ratio, MIN_KEEP_ABS)
        raw_keep = min(raw_keep, out_ch)

        if FRIENDLY_MULT > 1 and out_ch >= FRIENDLY_MULT:
            if FRIENDLY_ROUND == "up":
                min_keep = ceil_to_mult(raw_keep, FRIENDLY_MULT)
            else:
                min_keep = floor_to_mult(raw_keep, FRIENDLY_MULT)
            min_keep = max(min_keep, FRIENDLY_MULT)
            min_keep = min(min_keep, out_ch)
        else:
            min_keep = raw_keep

        max_prune = out_ch - min_keep
        note      = ""

    records.append({
        "layer_name":      nm,
        "group":           grp,
        "layer_type":      "ConvTranspose2d" if isinstance(mod, nn.ConvTranspose2d) else "Conv2d",
        "is_prunable":     is_prunable,
        "note":            note,
        "out_ch_baseline": out_ch,
        "min_keep_ch":     min_keep,
        "max_prune_ch":    max_prune,
    })

df_layers = pd.DataFrame(records)

# ── Parameter counting ────────────────────────────────────────────────────────
# NOTE: prev_eff_out is only updated for prunable layers.
# Transpose ups are skipped — their output channels are determined by the
# surrounding prunable convs, not by an independent pruning decision.
param_rows   = []
prev_eff_out = 3

for nm, mod in conv_layers:
    row         = df_layers[df_layers["layer_name"] == nm].iloc[0]
    min_keep    = int(row["min_keep_ch"])
    is_prunable = bool(row["is_prunable"])
    g           = int(getattr(mod, "groups", 1))

    if isinstance(mod, nn.Conv2d):
        _, _, Kh, Kw = mod.weight.shape
    else:
        _, _, Kh, Kw = mod.weight.shape

    p_base = int(mod.weight.numel()) + (int(mod.bias.numel()) if mod.bias is not None else 0)

    eff_in = prev_eff_out
    eff_out = min_keep
    p_min  = (eff_out * (eff_in // g) * Kh * Kw) + (min_keep if mod.bias is not None else 0)

    param_rows.append({
        "layer_name":         nm,
        "params_baseline":    p_base,
        "params_min_keep":    p_min,
        "params_removed_max": max(0, p_base - p_min),
    })

    # Only advance the channel tracker for prunable layers.
    # Unpruned transposes do not change the effective channel count
    # seen by the next prunable conv.
    if is_prunable:
        prev_eff_out = min_keep

df_params      = pd.DataFrame(param_rows)
df_theoretical = df_layers.merge(df_params, on="layer_name")

# ── FLOPs via hooks ───────────────────────────────────────────────────────────
def per_layer_macs(net: nn.Module, H: int, W: int, exclude_outc: bool) -> list[tuple[str, int]]:
    net.eval()
    ops, names = [], []
    for nm, m in net.named_modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            ops.append(m); names.append(nm)

    out_hw = {}

    def make_hook(nm_):
        def fn(module, inp, output):
            if isinstance(output, torch.Tensor) and output.ndim == 4:
                out_hw[nm_] = (int(output.shape[2]), int(output.shape[3]))
        return fn

    hooks = [m.register_forward_hook(make_hook(nm)) for nm, m in zip(names, ops)]
    with torch.no_grad():
        net(torch.zeros(1, 3, H, W))
    for h in hooks:
        h.remove()

    result = []
    for nm, m in zip(names, ops):
        if nm not in out_hw:
            continue
        if exclude_outc and nm.endswith("outc.conv"):
            result.append((nm, 0))
            continue
        Hout, Wout = out_hw[nm]
        g = int(getattr(m, "groups", 1))
        if isinstance(m, nn.Conv2d):
            Cout, Cin_g, Kh, Kw = m.weight.shape
        else:
            Cin, Cout_g, Kh, Kw = m.weight.shape
            Cout  = Cout_g * g
            Cin_g = Cin // g
        macs = Hout * Wout * int(Cout) * int(Cin_g) * Kh * Kw
        result.append((nm, int(macs)))
    return result

flops_mult = 2 if USE_2X_MACS_ else 1

# Baseline FLOPs
layer_macs_base = per_layer_macs(m_base, _H, _W, EXCLUDE_OUTC_)
macs_base       = sum(v for _, v in layer_macs_base)
flops_base_g    = macs_base * flops_mult / 1e9
print(f"Baseline FLOPs: {flops_base_g:.4f} GFLOPs  (input {_H}×{_W})")

# Exact min-keep FLOPs: build real UNet with min-keep channel vector
# Only pass channels for prunable layers; transpose ups are excluded from
# the channel file since they are not independently configured.
min_keep_vec = df_theoretical["min_keep_ch"].tolist()
ch_str       = "\n".join(str(c) for c in min_keep_vec)
flops_min_g  = None

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as tf:
    tf.write(ch_str)
    tmp_ch_path = tf.name

try:
    m_min = UNet(n_channels=3, n_classes=n_cl, f_channels=tmp_ch_path)
    m_min.eval()
    macs_min    = sum(v for _, v in per_layer_macs(m_min, _H, _W, EXCLUDE_OUTC_))
    flops_min_g = macs_min * flops_mult / 1e9
    del m_min
    print(f"Min-keep FLOPs (exact): {flops_min_g:.4f} GFLOPs")
except Exception as e:
    print(f"[WARN] Could not build min-keep model: {e}\n       Falling back to proportional estimate.")
finally:
    os.unlink(tmp_ch_path)

# Estimated min-keep FLOPs: proportional scaling per layer
# Transpose ups are skipped in the ratio chain (same reason as param counting).
layer_macs_min_est = []
prev_base_ch = 3
prev_min_ch  = 3

for (nm, macs_b), row in zip(layer_macs_base, df_theoretical.itertuples()):
    if EXCLUDE_OUTC_ and nm.endswith("outc.conv"):
        layer_macs_min_est.append((nm, 0))
        continue

    is_prunable = bool(row.is_prunable)
    out_b = int(row.out_ch_baseline)
    out_m = int(row.min_keep_ch)

    if is_prunable:
        ratio = (out_m / max(out_b, 1)) * (prev_min_ch / max(prev_base_ch, 1))
        layer_macs_min_est.append((nm, int(macs_b * ratio)))
        prev_base_ch = out_b
        prev_min_ch  = out_m
    else:
        # Transpose up: not pruned, include full baseline MACs
        layer_macs_min_est.append((nm, int(macs_b)))

flops_min_g_est = sum(v for _, v in layer_macs_min_est) * flops_mult / 1e9

# Merge per-layer FLOPs into df_theoretical
macs_base_map = {nm: v for nm, v in layer_macs_base}
macs_min_map  = {nm: v for nm, v in layer_macs_min_est}

df_theoretical["flops_baseline_gf"] = df_theoretical["layer_name"].map(
    {k: v * flops_mult / 1e9 for k, v in macs_base_map.items()}
).fillna(0.0)

df_theoretical["flops_min_keep_gf"] = df_theoretical["layer_name"].map(
    {k: v * flops_mult / 1e9 for k, v in macs_min_map.items()}
).fillna(0.0)

# ── Per-layer percentages ─────────────────────────────────────────────────────
df_theoretical["keep_%"]            = (df_theoretical["min_keep_ch"]        / df_theoretical["out_ch_baseline"]                          * 100).round(1)
df_theoretical["prune_%"]           = (df_theoretical["max_prune_ch"]       / df_theoretical["out_ch_baseline"]                          * 100).round(1)
df_theoretical["params_keep_%"]     = (df_theoretical["params_min_keep"]    / df_theoretical["params_baseline"].replace(0, np.nan)       * 100).round(1)
df_theoretical["params_removed_%"]  = (df_theoretical["params_removed_max"] / df_theoretical["params_baseline"].replace(0, np.nan)       * 100).round(1)
df_theoretical["flops_keep_%"]      = (df_theoretical["flops_min_keep_gf"]  / df_theoretical["flops_baseline_gf"].replace(0, np.nan)     * 100).round(1)
df_theoretical["flops_removed_%"]   = (100 - df_theoretical["flops_keep_%"]).round(1)

# ── Display per-layer table ───────────────────────────────────────────────────
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.4f}".format)

print("\n" + "="*110)
print("THEORETICAL PRUNING ANALYSIS — per layer")
print(f"  MIN_KEEP_RATIO={MIN_KEEP_RATIO}, MIN_KEEP_ABS={MIN_KEEP_ABS}, "
      f"FRIENDLY_MULT={FRIENDLY_MULT}, FRIENDLY_ROUND='{FRIENDLY_ROUND}'")
print(f"  Input: {_H}×{_W}  |  USE_2X_MACS={USE_2X_MACS_}  |  EXCLUDE_OUTC={EXCLUDE_OUTC_}")
print("="*110)

display(
    df_theoretical[[
        "layer_name", "group", "layer_type", "is_prunable", "note",
        "out_ch_baseline", "min_keep_ch",      "keep_%",
        "max_prune_ch",    "prune_%",
        "params_baseline", "params_min_keep",  "params_keep_%",
        "params_removed_max", "params_removed_%",
        "flops_baseline_gf", "flops_min_keep_gf", "flops_keep_%", "flops_removed_%",
    ]]
)

# ── Summary scalars ───────────────────────────────────────────────────────────
total_params_base    = int(sum(p.numel() for p in m_base.parameters()))
total_params_min_est = int(df_theoretical["params_min_keep"].sum())
total_params_removed = total_params_base - total_params_min_est
pct_params_kept      = 100 * total_params_min_est / total_params_base
pct_params_removed   = 100 * total_params_removed / total_params_base

# Prunable-only channel totals (excludes transpose ups and outc)
prunable_mask        = df_theoretical["is_prunable"]
total_ch_prunable    = int(df_theoretical.loc[prunable_mask, "out_ch_baseline"].sum())
total_ch_max_prune   = int(df_theoretical.loc[prunable_mask, "max_prune_ch"].sum())

print("\n" + "="*90)
print("SUMMARY")
print("="*90)
print(f"  Baseline params            : {total_params_base:>12,}  (100.0%)")
print(f"  Min-keep params (est.)     : {total_params_min_est:>12,}  ({pct_params_kept:.1f}% of original)")
print(f"  Max params removed (est.)  : {total_params_removed:>12,}  ({pct_params_removed:.1f}% of original)")
print()
print(f"  Baseline FLOPs             : {flops_base_g:>10.4f} GFLOPs  (100.0%)")
if flops_min_g is not None:
    pct_flops_kept_exact    = 100 * flops_min_g / flops_base_g
    pct_flops_removed_exact = 100 - pct_flops_kept_exact
    print(f"  Min-keep FLOPs (exact)     : {flops_min_g:>10.4f} GFLOPs  "
          f"({pct_flops_kept_exact:.1f}% of original, -{pct_flops_removed_exact:.1f}%)")
pct_flops_kept_est    = 100 * flops_min_g_est / flops_base_g
pct_flops_removed_est = 100 - pct_flops_kept_est
print(f"  Min-keep FLOPs (est.)      : {flops_min_g_est:>10.4f} GFLOPs  "
      f"({pct_flops_kept_est:.1f}% of original, -{pct_flops_removed_est:.1f}%)")
print()
print(f"  Max channels prunable      : {total_ch_prunable:>8}  (prunable layers only, excl. transpose ups + outc)")
print(f"  Max channels pruned        : {total_ch_max_prune:>8}  "
      f"({100*total_ch_max_prune/total_ch_prunable:.1f}% of prunable channels)")
print("="*90)

# ── Group-level breakdown ─────────────────────────────────────────────────────
grp = df_theoretical.groupby("group").agg(
    layers            = ("layer_name",           "count"),
    ch_baseline       = ("out_ch_baseline",       "sum"),
    ch_min_keep       = ("min_keep_ch",           "sum"),
    ch_max_prune      = ("max_prune_ch",          "sum"),
    params_baseline   = ("params_baseline",       "sum"),
    params_min_keep   = ("params_min_keep",       "sum"),
    params_removed    = ("params_removed_max",    "sum"),
    flops_base_gf     = ("flops_baseline_gf",     "sum"),
    flops_min_gf      = ("flops_min_keep_gf",     "sum"),
).copy()

grp["params_baseline_%"] = (grp["params_baseline"] / grp["params_baseline"].sum()                * 100).round(1)
grp["flops_base_%"]      = (grp["flops_base_gf"]   / grp["flops_base_gf"].sum()                  * 100).round(1)
grp["ch_prune_%"]        = (grp["ch_max_prune"]    / grp["ch_baseline"]                           * 100).round(1)
grp["params_keep_%"]     = (grp["params_min_keep"] / grp["params_baseline"].replace(0, np.nan)    * 100).round(1)
grp["params_remove_%"]   = (grp["params_removed"]  / grp["params_baseline"].replace(0, np.nan)    * 100).round(1)
grp["flops_keep_%"]      = (grp["flops_min_gf"]    / grp["flops_base_gf"].replace(0, np.nan)      * 100).round(1)
grp["flops_remove_%"]    = (100 - grp["flops_keep_%"]).round(1)

print("\nGroup-level breakdown:")
display(grp[[
    "layers",
    "ch_baseline",     "ch_min_keep",       "ch_max_prune",    "ch_prune_%",
    "params_baseline", "params_baseline_%",  "params_min_keep", "params_removed",  "params_keep_%", "params_remove_%",
    "flops_base_gf",   "flops_base_%",       "flops_min_gf",    "flops_keep_%",    "flops_remove_%",
]])

Baseline FLOPs: 436.8452 GFLOPs  (input 512×512)
[WARN] Could not build min-keep model: Given transposed=1, weight of size [512, 16, 2, 2], expected input[1, 16, 32, 32] to have 512 channels, but got 16 channels instead
       Falling back to proportional estimate.

THEORETICAL PRUNING ANALYSIS — per layer
  MIN_KEEP_RATIO=0.0, MIN_KEEP_ABS=1, FRIENDLY_MULT=16, FRIENDLY_ROUND='up'
  Input: 512×512  |  USE_2X_MACS=True  |  EXCLUDE_OUTC=True


,layer_name,group,layer_type,is_prunable,note,out_ch_baseline,min_keep_ch,keep_%,max_prune_ch,prune_%,params_baseline,params_min_keep,params_keep_%,params_removed_max,params_removed_%,flops_baseline_gf,flops_min_keep_gf,flops_keep_%,flops_removed_%
0,inc.double_conv.0,encoder,Conv2d,True,,64,16,25.0000,48,75.0000,1728,432,25.0000,1296,75.0000,0.9060,0.2265,25.0000,75.0000
1,inc.double_conv.3,encoder,Conv2d,True,,64,16,25.0000,48,75.0000,36864,2304,6.2000,34560,93.8000,19.3274,1.2080,6.2000,93.8000
2,down1.maxpool_conv.1.double_conv.0,encoder,Conv2d,True,,128,16,12.5000,112,87.5000,73728,2304,3.1000,71424,96.9000,9.6637,0.3020,3.1000,96.9000
3,down1.maxpool_conv.1.double_conv.3,encoder,Conv2d,True,,128,16,12.5000,112,87.5000,147456,2304,1.6000,145152,98.4000,19.3274,0.3020,1.6000,98.4000
4,down2.maxpool_conv.1.double_conv.0,encoder,Conv2d,True,,256,16,6.2000,240,93.8000,294912,2304,0.8000,292608,99.2000,9.6637,0.0755,0.8000,99.2000
5,down2.maxpool_conv.1.double_conv.3,encoder,Conv2d,True,,256,16,6.2000,240,93.8000,589824,2304,0.4000,587520,99.6000,19.3274,0.0755,0.4000,99.6000
6,down3.maxpool_conv.1.double_conv.0,encoder,Conv2d,True,,512,16,3.1000,496,96.9000,1179648,2304,0.2000,1177344,99.8000,9.6637,0.0189,0.2000,99.8000
7,down3.maxpool_conv.1.double_conv.3,encoder,Conv2d,True,,512,16,3.1000,496,96.9000,2359296,2304,0.1000,2356992,99.9000,19.3274,0.0189,0.1000,99.9000
8,down4.maxpool_conv.1.double_conv.0,bottleneck,Conv2d,True,,1024,16,1.6000,1008,98.4000,4718592,2304,0.0000,4716288,100.0000,9.6637,0.0047,0.0000,100.0000
9,down4.maxpool_conv.1.double_conv.3,bottleneck,Conv2d,True,,1024,16,1.6000,1008,98.4000,9437184,2304,0.0000,9434880,100.0000,19.3274,0.0047,0.0000,100.0000



SUMMARY
  Baseline params            :   31,037,698  (100.0%)
  Min-keep params (est.)     :      102,034  (0.3% of original)
  Max params removed (est.)  :   30,935,664  (99.7% of original)

  Baseline FLOPs             :   436.8452 GFLOPs  (100.0%)
  Min-keep FLOPs (est.)      :    74.1647 GFLOPs  (17.0% of original, -83.0%)

  Max channels prunable      :     5888  (prunable layers only, excl. transpose ups + outc)
  Max channels pruned        :     5600  (95.1% of prunable channels)

Group-level breakdown:


,layers,ch_baseline,ch_min_keep,ch_max_prune,ch_prune_%,params_baseline,params_baseline_%,params_min_keep,params_removed,params_keep_%,params_remove_%,flops_base_gf,flops_base_%,flops_min_gf,flops_keep_%,flops_remove_%
group,,,,,,,,,,,,,,,,
bottleneck,2,2048,32,2016,98.4000,14155776,45.6000,4608,14151168,0.0000,100.0000,28.9910,6.6000,0.0094,0.0000,100.0000
decoder,12,2880,1088,1792,62.2000,12186560,39.3000,80832,12105728,0.7000,99.3000,300.6477,68.8000,71.9281,23.9000,76.1000
encoder,8,1920,128,1792,93.3000,4683456,15.1000,16560,4666896,0.4000,99.6000,107.2064,24.5000,2.2272,2.1000,97.9000
other,1,2,2,0,0.0000,130,0.0000,34,96,26.2000,73.8000,0.0000,0.0000,0.0000,NaN,NaN
